In [1]:
# predict total score in any given nhl game

In [2]:
import pandas as pd


In [3]:
# write final modeling data to excel
modeling_data = pd.read_excel(r'data/modeling_data.xlsx', header=0)

# inspect
modeling_data.info()
modeling_data.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3405 entries, 0 to 3404
Data columns (total 33 columns):
 #   Column                            Non-Null Count  Dtype         
---  ------                            --------------  -----         
 0   Date                              3405 non-null   datetime64[ns]
 1   Season                            3405 non-null   int64         
 2   Game_ID                           3405 non-null   object        
 3   Home_Team                         3405 non-null   object        
 4   Away_Team                         3405 non-null   object        
 5   Total_Score                       3405 non-null   int64         
 6   Month                             3405 non-null   object        
 7   Day_of_Week                       3405 non-null   object        
 8   Conf_Matchup                      3405 non-null   bool          
 9   Div_Matchup                       3405 non-null   bool          
 10  Conf_Pair                         3310 non-null 

,Date,Season,Game_ID,Home_Team,Away_Team,Total_Score,Month,Day_of_Week,Conf_Matchup,Div_Matchup,...,prop_Reg_Home_Win_Away,prop_Reg_Away_Win_Away,prop_Reg_Tie_Away,prop_FT_Home_Win_Away,prop_FT_Away_Win_Away,avg_FT_home_goals_per_game_Away,avg_FT_away_goals_per_game_Away,prop_reg_home_goal_diff_Away,avg_reg_home_goals_per_game_Away,avg_reg_away_goals_per_game_Away
0,2023-09-24,2023,2023-09-24-20:00|Anaheim Ducks vs Los Angeles ...,Anaheim Ducks,Los Angeles Kings,5,September,Sunday,True,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-09-27,2023,2023-09-27-22:00|Anaheim Ducks vs San Jose Sharks,Anaheim Ducks,San Jose Sharks,6,September,Wednesday,True,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023-09-29,2023,2023-09-29-22:00|Anaheim Ducks vs Los Angeles ...,Anaheim Ducks,Los Angeles Kings,7,September,Friday,True,True,...,0.5,0.0,0.5,1.0,0.0,4.000000,2.500000,1.400000,3.500000,2.500000
3,2023-10-05,2023,2023-10-05-22:00|Anaheim Ducks vs Arizona Coyotes,Anaheim Ducks,Arizona Coyotes,6,October,Thursday,False,False,...,1.0,0.0,0.0,1.0,0.0,4.333333,1.333333,3.250000,4.333333,1.333333
4,2023-10-15,2023,2023-10-15-20:30|Anaheim Ducks vs Carolina Hur...,Anaheim Ducks,Carolina Hurricanes,9,October,Sunday,False,False,...,1.0,0.0,0.0,1.0,0.0,4.333333,1.000000,4.333333,4.333333,1.000000


In [4]:
# set season as categorical
modeling_data['Season'] = modeling_data['Season'].astype('str')
modeling_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3405 entries, 0 to 3404
Data columns (total 33 columns):
 #   Column                            Non-Null Count  Dtype         
---  ------                            --------------  -----         
 0   Date                              3405 non-null   datetime64[ns]
 1   Season                            3405 non-null   object        
 2   Game_ID                           3405 non-null   object        
 3   Home_Team                         3405 non-null   object        
 4   Away_Team                         3405 non-null   object        
 5   Total_Score                       3405 non-null   int64         
 6   Month                             3405 non-null   object        
 7   Day_of_Week                       3405 non-null   object        
 8   Conf_Matchup                      3405 non-null   bool          
 9   Div_Matchup                       3405 non-null   bool          
 10  Conf_Pair                         3310 non-null 

In [5]:
modeling_data['Total_Score'].describe()

count    3405.000000
mean        6.129515
std         2.290960
min         1.000000
25%         5.000000
50%         6.000000
75%         7.000000
max        17.000000
Name: Total_Score, dtype: float64

In [6]:
# drop na to make life easier
modeling_data = modeling_data.dropna()

# trim data roughly to regular season games only
modeling_data = modeling_data[
    (modeling_data['Month'].isin(['October', 'November', 'December', 'January', 'February', 'March'])) # trim to regular season months with a smidge of oct pre-season
                              | 
    (modeling_data['Month'].isin(['April']) & modeling_data['Date'].dt.day <= 15) # include up to april 15th
]

# inspect
modeling_data.info()
modeling_data['Total_Score'].describe()

<class 'pandas.core.frame.DataFrame'>
Index: 3065 entries, 2 to 3404
Data columns (total 33 columns):
 #   Column                            Non-Null Count  Dtype         
---  ------                            --------------  -----         
 0   Date                              3065 non-null   datetime64[ns]
 1   Season                            3065 non-null   object        
 2   Game_ID                           3065 non-null   object        
 3   Home_Team                         3065 non-null   object        
 4   Away_Team                         3065 non-null   object        
 5   Total_Score                       3065 non-null   int64         
 6   Month                             3065 non-null   object        
 7   Day_of_Week                       3065 non-null   object        
 8   Conf_Matchup                      3065 non-null   bool          
 9   Div_Matchup                       3065 non-null   bool          
 10  Conf_Pair                         3065 non-null   obj

count    3065.000000
mean        6.134421
std         2.303124
min         1.000000
25%         5.000000
50%         6.000000
75%         7.000000
max        17.000000
Name: Total_Score, dtype: float64

In [7]:
# designate response variable
response_ = 'Total_Score'


In [8]:
# find ideal sample size to test on all of 2025
samp_size_2025 = modeling_data[modeling_data['Season']=='2025'].shape[0] / modeling_data.shape[0]
print(f'Sample size for 2025 season: {samp_size_2025:.2%}')

modeling_data['Season'].value_counts()

Sample size for 2025 season: 9.46%


Season
2024    1425
2023    1350
2025     290
Name: count, dtype: int64

In [14]:
# create test and train data
test_start_date = '2025-11-07'

# list of drop cols that won't be used in modeling
drop_cols = ['Game_ID', 'Date']

# isolate train
train_data = modeling_data[modeling_data['Date'] < test_start_date].drop(columns=drop_cols, axis=1)

# isolate test
test_data = modeling_data[modeling_data['Date'] >= test_start_date].drop(columns=drop_cols + [response_], axis=1)

# inspect
train_data.info()


<class 'pandas.core.frame.DataFrame'>
Index: 3027 entries, 2 to 3404
Data columns (total 31 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Season                            3027 non-null   object 
 1   Home_Team                         3027 non-null   object 
 2   Away_Team                         3027 non-null   object 
 3   Total_Score                       3027 non-null   int64  
 4   Month                             3027 non-null   object 
 5   Day_of_Week                       3027 non-null   object 
 6   Conf_Matchup                      3027 non-null   bool   
 7   Div_Matchup                       3027 non-null   bool   
 8   Conf_Pair                         3027 non-null   object 
 9   Div_Pair                          3027 non-null   object 
 10  Team_Pair                         3027 non-null   object 
 11  prop_Reg_Home_Win_Home            3027 non-null   float64
 12  prop_Reg_Aw

In [15]:
# import bambi
import bambi as bmb

# 1. Inspect Total_Score distribution
mu_runs = modeling_data['Total_Score'].mean()
sd_runs = modeling_data['Total_Score'].std()

print('mu_runs', mu_runs)
print('sd_runs', sd_runs)
print('var_runs', sd_runs ** 2)

# 3. Estimate overdispersion (only for Negative Binomial)
dispersion_alpha_est = (sd_runs**2 - mu_runs) / (mu_runs**2)
dispersion_alpha_est = max(dispersion_alpha_est, 1e-3)  # Avoid zero or negative
print('dispersion_alpha_est', dispersion_alpha_est)

priors_ = {
    # Overdispersion prior for Negative Binomial
    "alpha": bmb.Prior("Exponential", lam=1 / dispersion_alpha_est)
}

mu_runs 6.13442088091354
sd_runs 2.3031238433232266
var_runs 5.30437943768395
dispersion_alpha_est 0.001


In [16]:
# check avail cols
train_data.columns

Index(['Season', 'Home_Team', 'Away_Team', 'Total_Score', 'Month',
       'Day_of_Week', 'Conf_Matchup', 'Div_Matchup', 'Conf_Pair', 'Div_Pair',
       'Team_Pair', 'prop_Reg_Home_Win_Home', 'prop_Reg_Away_Win_Home',
       'prop_Reg_Tie_Home', 'prop_FT_Home_Win_Home', 'prop_FT_Away_Win_Home',
       'avg_FT_home_goals_per_game_Home', 'avg_FT_away_goals_per_game_Home',
       'prop_reg_home_goal_diff_Home', 'avg_reg_home_goals_per_game_Home',
       'avg_reg_away_goals_per_game_Home', 'prop_Reg_Home_Win_Away',
       'prop_Reg_Away_Win_Away', 'prop_Reg_Tie_Away', 'prop_FT_Home_Win_Away',
       'prop_FT_Away_Win_Away', 'avg_FT_home_goals_per_game_Away',
       'avg_FT_away_goals_per_game_Away', 'prop_reg_home_goal_diff_Away',
       'avg_reg_home_goals_per_game_Away', 'avg_reg_away_goals_per_game_Away'],
      dtype='object')

In [ ]:
# Hierarchical Bayesian NegBin model: random effects for teams
import arviz as az
import multiprocessing

    # "Total_Score ~ 1 + (1|Division_Interaction) + DayOfWeek * time_of_day + Month + Year + (1|Home_Team) + (1|Away_Team) + \
    #  (1|Team_Interaction) + Stadium_Indoor + Coors_Field + (1|Home_Team:Year) + (1|Away_Team:Year) + (1|Team_Interaction:Year) + \
    #  (1|Home_Cluster) +  (1|Away_Cluster) + (1|Home_Cluster:Away_Cluster)",

# set model fmla, train data, and model family
hierarchical_model = bmb.Model(
    "Total_Score ~ 1 + Day_of_Week + Month + Season + (1|Home_Team) + (1|Away_Team) + \
     (1|Team_Pair) + (1|Home_Team:Season) + (1|Away_Team:Season) + (1|Team_Pair:Season) + \
     Conf_Matchup + Div_Matchup + Conf_Pair + Div_Pair",
    train_data,
    family="negativebinomial",
    priors=priors_
)

# set chains and cores
n_chains = 3
n_cores = multiprocessing.cpu_count()

# set tune and draw size
tune_size = int(train_data.shape[0] * 0.33) # set tune to X% of all games
draw_size = 2 * tune_size # set draws to be double the tune size
print(f'tune_size: {tune_size}')
print(f'draw_size: {draw_size}')

# Fit the hierarchical model
hierarchical_results = hierarchical_model.fit(
    tune=tune_size,          # tuning (warm-up) steps before real sampling; 1500
    draws=draw_size,         # real samples after tuning; total samples = draws x chains; 3000
    target_accept=0.99, # higher value yields better quality
    chains=n_chains,           # run X separate MCMC chains; Ensures good convergence diagnostics and higher total ESS
    cores=n_cores,            # use X CPU cores (one per chain)
    random_seed=42,     # fix randomness for reproducibility
    init="jitter+adapt_diag",  # can change to "jitter+adapt_diag" for better initialization; old = 'adapt_diag'
    progressbar=True    # show a progress bar while fitting
)

# store summary as a df and inspect
summary_df = az.summary(hierarchical_results)
summary_df.head()


tune_size: 998
draw_size: 1996


Initializing NUTS using jitter+adapt_diag...
C:\Users\edrak\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\pytensor\tensor\rewriting\elemwise.py:887: UserWarning: Loop fusion failed because the resulting node would exceed the kernel argument limit.
  warn(
Multiprocess sampling (3 chains in 3 jobs)
NUTS: [alpha, Intercept, Day_of_Week, Month, Season, Conf_Matchup, Div_Matchup, Conf_Pair, Div_Pair, 1|Home_Team_sigma, 1|Home_Team_offset, 1|Away_Team_sigma, 1|Away_Team_offset, 1|Team_Pair_sigma, 1|Team_Pair_offset, 1|Home_Team:Season_sigma, 1|Home_Team:Season_offset, 1|Away_Team:Season_sigma, 1|Away_Team:Season_offset, 1|Team_Pair:Season_sigma, 1|Team_Pair:Season_offset]


C:\Users\edrak\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\
Python312\site-packages\rich\live.py:256: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

In [ ]:
# plot trace
az.plot_trace(hierarchical_results)

In [ ]:
# plot energy
az.plot_energy(hierarchical_results)

In [11]:
# import joblib

# # write model to pkl file
# joblib.dump(basic_model, 'model/model_total_score.pkl')
